In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# 1. Import Libraries


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import make_scorer, mean_squared_error
import lightgbm as lgb
from scipy.stats import uniform, randint
from sklearn.model_selection import RandomizedSearchCV, KFold
from catboost import CatBoostRegressor
import xgboost as xgb

# 2. Import Train-Test-Submission_Sample Files

In [ ]:
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
sub = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv")



# 3. EDA

In [ ]:
print(f"df Train shape {train.shape}")
print(f"df Test shape {test.shape}")

In [ ]:
#lets see each columns min and max values
train.describe().T

In [ ]:
pd.set_option('display.max_columns', None)
train.tail()

In [ ]:
num_cols = train.select_dtypes(exclude= 'object').columns
print('Numerical columns :' ,num_cols ,"\n\n number of numerical columns:" ,len(num_cols))

print("="*100)
cat_cols = train.select_dtypes(include= 'object').columns
print('Categorical columns :' ,cat_cols,"\n\n number of categorical columns:" ,len(cat_cols) )

## 3.1. Correlation 

In [ ]:
plt.figure(figsize=(15,10))
#Mask upper triangle to reduce clutter (show each pair once)
mask = np.triu(np.ones_like(train[train.select_dtypes(exclude= 'object').columns].corr(), dtype=bool))
sns.heatmap(train[train.select_dtypes(exclude= 'object').columns].corr(),
            mask=mask,
           annot=True,
           cmap='Greens',
           fmt=".1f")

plt.title("Correlation Matrix ")
plt.tight_layout()
plt.show()

# 4. Preprocess
combine train and test for preprocessing 

In [ ]:
train_ids = train['Id']
test_ids = test['Id']
train_target = train['SalePrice']

train_features = train.drop(['Id', 'SalePrice'], axis=1)
test_features = test.drop('Id', axis=1)

combine_data = pd.concat([train_features, test_features], axis=0, ignore_index=True)

## 4.1 Check and Handle Missing  Values


In [ ]:
#Check Null Value
combine_data.isna().sum().sum()

In [ ]:
cols_with_miss = [col for col in combine_data.columns 
                                 if combine_data[col].isnull().any()]

In [ ]:
#Create dataframe of missing value columns

miss_val_df = pd.DataFrame(combine_data[cols_with_miss].isna().sum(), columns=['Miss value'])
miss_val_df

### **i. Categorize Missings**



**There is 3 types of Missing value here :**

**1. Missing for a Reason (None): These should be filled with a placeholder like "None" or 0.**

   PoolQC: Missing = No Pool

   MiscFeature: Missing = No miscellaneous feature

   Alley: Missing = No alley access

   Fence: Missing = No fence

   FireplaceQu: Missing = No fireplace

   GarageType, GarageFinish, GarageQual, GarageCond: Missing = No garage

   BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2: Missing = No basement

   MasVnrType: Missing = No masonry veneer (use "None")

   MasVnrArea: Missing = No masonry veneer (use 0)

   


**2. The Special Case:**

 **i : LotFrontage**

   Impute by grouping by Neighborhood and taking the median LotFrontage for that group. This is much better than using the overall median.

 **ii : GarageYrBlt**

   Fill with 0 -1 or YearBuilt

**3. Truly Missing (Likely Data Entry Errors): These require imputation based on similar houses (mode, median, or mean).**


### **ii. Missing Values With Reason**

In [ ]:
cols_miss_with_reason = ['Alley','PoolQC', 'MiscFeature', 'Fence','FireplaceQu','GarageType','GarageFinish','GarageQual',
                      'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure','BsmtFinType1', 'BsmtFinType2', 'MasVnrType',
                     ]

#Fill with "None" for categorical
for col in cols_miss_with_reason :
   combine_data[col] = combine_data[col].fillna('None')


#Fill with 0 for 'MasVnrArea'
combine_data['MasVnrArea'] = combine_data['MasVnrArea'].fillna(0)


### **ii. The Special Case 1: LotFrontage**


* It's linear feet of street connected to the property.
* It's likely "truly missing" but has a high correlation with Neighborhood and LotArea.


**Best : Impute by grouping by Neighborhood and taking the median LotFrontage for that group. This is much better than using the overall median.**

In [ ]:
combine_data['LotFrontage'] = combine_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median()))


###  **ii. The Special Case 2: GarageYrBlt**


* Missing values occur when there is no garage.
*  The year is missing because the garage doesn't exist.

** Impute by -1 or 0 **
-1 to make it clearly different from actual years

In [ ]:
combine_data['GarageYrBlt'] = combine_data['GarageYrBlt'].fillna(-1)


### **iii. Truly Missing**

* *Impute by Mode for Categorical*
* *Impute by Median for Numerical*

In [ ]:
cols_truly_miss_cat = combine_data.select_dtypes(include= 'object').columns

for col in cols_truly_miss_cat :
    combine_data[col] = combine_data[col].fillna(combine_data[col].mode()[0])


In [ ]:
cols_truly_miss_num = combine_data.select_dtypes(exclude= 'object').columns

for col in cols_truly_miss_num :
    combine_data[col] = combine_data[col].fillna(combine_data[col].median())

### **iv. Finale check if Any missing left**

In [ ]:
combine_data.isna().sum().sum()

## **4.2 Feature Engineering**


### **4.2.1 Total Square Footage**
 Total Square Footage :  One comprehensive feature that represents total house size
 
 **Why this works:**

* Buyers care about total living space, not individual components

* Combines basement, 1st floor, and 2nd floor into one meaningful metric

* Often has a stronger correlation with SalePrice than any individual component

* Handles houses with different configurations (e.g., large basement vs. two-story)

In [ ]:
combine_data['TotalSF'] = combine_data['TotalBsmtSF'] + combine_data['1stFlrSF'] + combine_data['2ndFlrSF']

### **4.2.2  Total Bathrooms**

**Why this works:**

* Real estate markets value bathrooms in a standardized way (full bath = 1, half bath = 0.5)

*  Combines 4 separate bathroom counts into one logical metric

*   Much more interpretable for models than separate counts

*   Reflects how appraisers and buyers actually think about bathroom count

**Example: A house with 2 full baths + 1 half bath = 2.5 total bathrooms**


In [ ]:
combine_data['TotalBath'] = (combine_data['FullBath'] + 
                        (0.5 * combine_data['HalfBath']) + 
                        combine_data['BsmtFullBath'] + 
                        (0.5 * combine_data['BsmtHalfBath']))

### **4.2.3  House Age Features (HouseAge, RemodAge)**

**Why this works:**

*    Age is more important than absolute year for pricing

*    Newer houses typically sell for more, all else being equal

*    RemodAge tells you how long ago the house was updated (critical for older homes)

*    A 50-year-old house that was remodeled 2 years ago is very different from one never remodeled

**Business Insight: Buyers pay premiums for recently updated properties**


In [ ]:
combine_data['HouseAge'] = combine_data['YrSold'] - combine_data['YearBuilt']
combine_data['RemodAge'] = combine_data['YrSold'] - combine_data['YearRemodAdd']

### **4.2.4  Quality × Size Interaction**
**Why this works:**

*    Interaction effects - quality and size multiply each other's impact

*    A large, high-quality house is worth more than the sum of its parts

*    A small, high-quality house might be over-improved for its neighborhood

*    Captures the non-linear relationship between quality and price

**Example:**

*    2000 sqft × Quality 8 = 16,000 (premium house)

*    2000 sqft × Quality 3 = 6,000 (fixer-upper)

In [ ]:
combine_data['OverallQual_TotalSF'] = combine_data['OverallQual'] * combine_data['TotalSF']


## **4.3 Outlier Detection and Handeling**

### 4.3.1 Visual Detection

In [ ]:
fig, axs = plt.subplots(ncols=4, nrows=10, figsize=(10, 20))
index = 0
axs = axs.flatten()
for k,v in train[num_cols].items():
    sns.boxplot(y=k, data=train[num_cols], ax=axs[index])
    index += 1
plt.tight_layout(pad=0.4, w_pad=0.5, h_pad=5.0)


In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.scatter(train['GrLivArea'], train['SalePrice'], alpha=0.6)
plt.xlabel('Above Grade Living Area (sq ft)')
plt.ylabel('Sale Price')
plt.title('GrLivArea vs SalePrice - Look for points in bottom right')

# Add some guidelines to spot outliers
plt.axvline(x=4000, color='red', linestyle='--', alpha=0.7, label='Potential outlier threshold')
plt.legend()